# 01 — Exploratory Data Analysis: CBIS-DDSM

Quick sanity checks on the dataset before training:
1. Class balance (benign vs malignant)
2. Image dimensions and intensity distributions
3. Sample visualizations before and after preprocessing

Run from the project root so the `src.` imports resolve.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import build_dataframe, preprocess_mammogram

with open('../configs/config.yaml') as f:
    config = yaml.safe_load(f)
config

In [ ]:
raw_dir = '../' + config['dataset']['raw_dir']
df = build_dataframe(f'{raw_dir}/metadata.csv', raw_dir)
df.head()

## Class balance

In [ ]:
counts = df['label'].value_counts().rename({0: 'Benign', 1: 'Malignant'})
print(counts)
counts.plot.bar(color=['#4C72B0', '#C44E52'])
plt.title('Class distribution')
plt.ylabel('Count')
plt.show()

## Sample images: raw vs preprocessed

In [ ]:
from src.data.preprocessing import read_image, crop_to_breast, apply_clahe

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
samples = df.sample(4, random_state=42)

for i, (_, row) in enumerate(samples.iterrows()):
    raw = read_image(row['abs_path'])
    cropped = crop_to_breast(raw)
    enhanced = apply_clahe(cropped)
    label_name = 'Malignant' if row['label'] == 1 else 'Benign'

    axes[0, i].imshow(raw, cmap='gray')
    axes[0, i].set_title(f'Raw — {label_name}')
    axes[1, i].imshow(cropped, cmap='gray')
    axes[1, i].set_title('Cropped')
    axes[2, i].imshow(enhanced, cmap='gray')
    axes[2, i].set_title('+ CLAHE')
    for ax in axes[:, i]:
        ax.axis('off')

plt.tight_layout()
plt.show()